# Hypotheses and decisions not promoted to the main protocol

This is a research register, not a list of proven improvements. No remote accuracy experiment has been run for this refactor. Historical test-set observations can motivate hypotheses but cannot validate them. Run alternatives on identical dev IDs, select one primary metric and request budget, then evaluate the chosen design on a fresh holdout.

`root` here means the initial choice among graph roots. The separate question of **including the current parent CWE** alongside its children concerns stopping, not whether roots are visited.

In [ ]:
from collections import Counter
from pathlib import Path

from classification import CWEGraph
from experiment import default_config

graph = CWEGraph(Path.cwd() / "data" / "all.csv")
branch_sizes = {
    label: len(children) + 1 for label, children in graph.children.items() if children
}
{
    "catalog_size": len(graph.labels),
    "root_options": len(graph.roots),
    "largest_child_plus_parent": max(branch_sizes.values(), default=0),
    "root_abstractions": dict(
        Counter(
            graph.labels[label].get("Weakness Abstraction") for label in graph.roots
        )
    ),
    "flat_fits_decisions": len(graph.labels) <= 255,
}

## H1: skip roots and classify directly

**Motivation:** an abstract root name may not communicate its descendants. **Risk:** removing roots changes the label space. Removing only root labels does not remove routing; offering every nonroot is a different flat classifier and may exceed 255 choices. Do not split labels into arbitrary 255-label batches and compare their raw choice probabilities: normalization depends on the alternatives in each question.

A fair alternative retrieves a fixed shortlist, then asks one choice question with an explicit none-of-the-above option. Report retrieval recall@N, oracle shortlist ceiling, end-to-end hit@1, macro F1 and tokens/requests. Retrieve from catalog descriptions only; never inject gold labels. Retain broad CWEs as possible correct answers, or report excluded-label coverage explicitly. This remains a **proposal**, not a production root-bypass switch.

## H2: parent CWE vs abstract STOP

A parent option carries a real definition; abstract STOP carries a negated instruction. Either can bias stopping. Compare these on matched dev examples with the same model and traversal; inspect overspecific stopping and wrong-root rates as well as accuracy. This adapter ablation is runnable below, but disabled by default. It produces separate outputs and does not update the main config. Do not tune the provider or prompt after seeing its test results.

In [ ]:
RUN_STOP_ABLATION = False
ablation = default_config(Path.cwd())
ablation.update(
    split="dev",
    max_examples=100,
    methods=["greedy", "beam"],
    output=str(Path.cwd() / "experiment" / "ablation-results" / "parent-vs-stop"),
    models={
        name: "experiment.decisions_selector:create_selector"
        for name in ("parent", "stop")
    },
    model_options={
        name: {"model": "gpt-6-luna", "stop_style": name} for name in ("parent", "stop")
    },
)
if RUN_STOP_ABLATION:
    from experiment.runner import run_experiment

    ablation_summary = run_experiment(ablation)
    display(ablation_summary["systems"])
else:
    print("Not run. Requires prepared dev benchmark and OPENAI_API_KEY.")

## H3: rich descendants and aliases

The original README reports gains after inspecting the old test set. That is hypothesis-generating evidence only. Compare plain definitions, aliases alone, and aliases plus descendant summaries on dev. Match option ordering, traversal, model and budget. Rank descendants with dev-only counts generated from the same benchmark; record their content hash. Watch head-label bias: rare descendants omitted from the summary may become invisible. Avoid frequency summaries from the full dataset or an unverified stale frequencies file.

Do not transfer historical Jev quality claims to OpenAI. The new OpenAI adapter uses plain definitions; rich descriptions remain an explicit legacy factory until a clean ablation justifies promotion.

## H4: path products, DAG mass and length normalization

The current beam uses the maximum path score for a node, not the sum of its alternate paths. Products favor short routes and parent stops. Local probabilities depend on the offered option set, so multiplying them is not a proof of calibrated end-to-end uncertainty. Summing arbitrary overlapping DAG paths can double count mass. Dividing by path length changes the ranking and invalidates the current early-stop bound (which assumes scores only decrease).

Before implementing a replacement, define a generative routing model or treat scores purely as ranking features. Use dev to compare the baseline with a reranker and inspect outcomes by depth; fit calibration on a separate dev partition, not on the final holdout. Choose coverage thresholds on calibration data and report frozen-threshold performance.

## H5: label mapping and larger samples

Do not automatically replace every gold CWE with a leaf: broad curator labels can be correct, and changing gold can manufacture gains. Keep exact-match as primary and hierarchical similarity as secondary. Review a blinded, stratified set of disagreements with two reviewers and retain adjudication notes.

Do not assume 2,000 old test examples repair test-set reuse. Collect a later snapshot/time window or an independent report source and group package families/aliases across boundaries. Frozen IDs, full labels, all exclusions and dev-derived artifacts must be retained. Measure rare-class support, ecosystem slices and realistic unlabeled/out-of-taxonomy reports before deployment.